In [1]:
# from urllib.robotparser import RobotFileParser
#
# robots_url = 'https://urbania.pe/robots.txt'
# parser = RobotFileParser()
# parser.set_url(robots_url)
# parser.read()
#
# parser.site_maps()

In [4]:
import os

os.chdir("C:/Users/maria/PycharmProjects/Urbania_Proyecto")

# Robots y sitemaps
1. Encontramos el robots, extraemos el txt y hallamos los sitemaps

In [5]:
import requests

robots_url = 'https://urbania.pe/robots.txt'
text = requests.get(robots_url).text

sitemaps_xml = [line.split(": ", 1)[1] for line in text.splitlines() if line.lower().startswith("sitemap:")]
print(sitemaps_xml)

['https://urbania.pe/sitemaps_https.xml', 'https://urbania.pe/sitemaps_list_emp_https.xml', 'https://urbania.pe/sitemaps_prop_emp_https.xml']


In [5]:
# sitemaps_xml = [url for url in parser.site_maps() if url.endswith(".xml")]
# print(sitemaps_xml)

In [6]:
import requests

url = "https://urbania.pe/sitemaps_https.xml"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/128.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Referer": "https://urbania.pe/",
    "Connection": "keep-alive"
}

resp = requests.get(url, headers=headers)
print(resp.status_code)
print(resp.text[:500])

403
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helve


Existen muchas capas de seguridad dentro del sitemaps, entonces, a pesar de ser permitido por los robots, es mejor descargar el xml y abrirlo en la carpeta de data

In [ ]:
#!pip install lxml

In [7]:
import lxml
print(lxml.__version__)

6.0.2


In [10]:
from bs4 import BeautifulSoup

# Ruta relativa al archivo
with open("Scrapping/data/sitemaps_https.xml", "rb") as f:
    xml_content = f.read()

# Parsear con BeautifulSoup
soup = BeautifulSoup(xml_content, "lxml-xml")

for url in soup.find_all("loc"): #[:5]
    print(url.text)

https://urbania.pe/sitemap_core_https.xml.gz
https://urbania.pe/sitemap_list_https_1.xml.gz
https://urbania.pe/sitemap_prop_https_1.xml.gz
https://urbania.pe/sitemap_im_https_1.xml.gz


In [18]:
import asyncio
import gzip
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def fetch_sitemap():
    url = "https://urbania.pe/sitemap_core_https.xml.gz"

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()

        response = await page.goto(url)
        if response.status != 200:
            raise Exception(f"Error HTTP: {response.status}")

        # Contenido binario del .gz
        gz_content = await response.body()

        # Descomprimir
        xml_content = gzip.decompress(gz_content)

        # Parsear con BeautifulSoup
        soup = BeautifulSoup(xml_content, "xml")

        # Mostrar primeras 5 URLs
        for loc in soup.find_all("loc")[:5]:
            print(loc.text)

        await browser.close()

# Ejecutar la corrutina en Jupyter
await fetch_sitemap()

NotImplementedError: 

In [16]:
print(resp.status_code)

403


In [7]:
import requests
from bs4 import BeautifulSoup

def urls_from_sitemap(xml_content, max_num_items=10):
    soup = BeautifulSoup(xml_content, "xml")
    data_dict = {}

    num_items = 0
    for url in soup.find_all('url'):
        if num_items >= max_num_items:
            break

        loc = url.find('loc')
        lastmod = url.find('lastmod')

        if loc:
            data_dict[loc.text] = lastmod.text if lastmod else None

        num_items += 1

    return data_dict

def sitemaps_from_sitemap_index(xml_content):
    soup = BeautifulSoup(xml_content, "xml")

    all_urls = []
    for _sitemap in soup.find_all('sitemap'):
        loc = _sitemap.find('loc')
        if loc:
            sitemap_resp = requests.get(loc.text)
            sitemap_resp.raise_for_status()

            sitemap_urls_dict = urls_from_sitemap(sitemap_resp.content)
            all_urls.extend(list(sitemap_urls_dict.keys()))

    return all_urls



In [8]:
urls_list = []
for sitemap in sitemaps_xml:
    resp = requests.get(sitemap)
    resp.raise_for_status()
    urls_list.extend(sitemaps_from_sitemap_index(resp.content))

print(urls_list)

HTTPError: 403 Client Error: Forbidden for url: https://urbania.pe/sitemaps_https.xml